# Countersign pool — metric selection (calibration refit → 4 distances compared, original channel only)
Runtime → Change runtime type → **T4 GPU**, then Runtime → Run all.
Everything is cached on Drive; if Colab disconnects, just Run all again.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, subprocess
DRIVE = '/content/drive/MyDrive/countersign'
assert os.path.exists(f'{DRIVE}/countersign_bundle.zip'), 'upload countersign_bundle.zip to MyDrive/countersign'
!rm -rf /content/cs && mkdir -p /content/cs && cd /content/cs && unzip -q -o {DRIVE}/countersign_bundle.zip
os.makedirs(f'{DRIVE}/countersign_runs', exist_ok=True)
# runs/ lives on Drive: caches and results survive disconnects
!cd /content/cs && cp -rn runs/. {DRIVE}/countersign_runs/ && rm -rf runs && ln -s {DRIVE}/countersign_runs runs
!ls /content/cs /content/cs/runs

In [ ]:
!pip -q install numba pyyaml openai-whisper edge-tts gTTS librosa soundfile statsmodels funasr modelscope kaldi-native-fbank
# stop Run all here if anything is missing
import whisper, funasr, edge_tts, gtts, numba, yaml, librosa, soundfile, statsmodels, kaldi_native_fbank
import torch; assert torch.cuda.is_available(), 'no GPU: Runtime -> Change runtime type -> T4 GPU'
print('OK, GPU:', torch.cuda.get_device_name(0))

In [ ]:
# trial/TTS cache on local disk (SQLite is unreliable on Drive), backed up to Drive every 5 min
os.environ['CS_CACHE'] = '/content/cache'
!mkdir -p /content/cache {DRIVE}/countersign_cache && cp -rn {DRIVE}/countersign_cache/. /content/cache/
# a journal copied from Drive would be replayed as a hot journal and could roll back / corrupt the cache
!rm -f /content/cache/trials.sqlite-journal /content/cache/trials.sqlite.tmp
if os.path.exists('/content/cache/trials.sqlite'):
    import sqlite3 as _s; _c = _s.connect('/content/cache/trials.sqlite')
    print('cache integrity:', _c.execute('PRAGMA integrity_check').fetchone()[0],
          '| cached trials:', _c.execute('SELECT count(*) FROM fc').fetchone()[0]); _c.close()
# backup = a consistent SQLite snapshot (small) + only NEW TTS files.  Recopying every TTS
# file each time flooded Drive's upload queue and the trial cache never reached Drive.
import sqlite3, shutil, threading, time
def backup():
    if os.path.exists('/content/cache/trials.sqlite'):
        src = sqlite3.connect('/content/cache/trials.sqlite'); dst = sqlite3.connect('/content/snap.sqlite')
        src.backup(dst); dst.close(); src.close()
        shutil.copyfile('/content/snap.sqlite', f'{DRIVE}/countersign_cache/trials.sqlite.tmp')
        os.replace(f'{DRIVE}/countersign_cache/trials.sqlite.tmp', f'{DRIVE}/countersign_cache/trials.sqlite')
    subprocess.run(['bash', '-c', f'cp -rn /content/cache/tts {DRIVE}/countersign_cache/ 2>/dev/null'])
    with open(f'{DRIVE}/countersign_cache/last_backup.txt', 'w') as f:
        f.write(time.strftime('%Y-%m-%d %H:%M:%S UTC', time.gmtime()))
def _loop():
    while True:
        time.sleep(300)
        try: backup()
        except Exception as e: print('backup failed:', e)
threading.Thread(target=_loop, daemon=True).start()
def step(script, *args):
    # a failing step raises, so Run all stops instead of running the next steps on missing files;
    # output is also appended to runs/logs/ (on Drive) so progress can be read remotely
    os.makedirs('/content/cs/runs/logs', exist_ok=True)
    name = os.path.basename(script)[:-3] + ''.join('_' + a.strip('-') for a in args)
    log = open(f'/content/cs/runs/logs/{name}.log', 'a')
    p = subprocess.Popen(['python', '-X', 'utf8', script, *args], cwd='/content/cs',
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=os.environ)
    for line in p.stdout:
        if 'it/s' not in line:
            print(line, end='', flush=True)
            log.write(line); log.flush()
    p.wait()
    log.close()
    backup()
    if p.returncode:
        raise RuntimeError(f'{script} failed (exit {p.returncode}); see the traceback above')

In [ ]:
step('scripts/s2_calibration.py', '--section', 'calibration_v2')

In [ ]:
step('scripts/s8_metric_selection.py')

In [ ]:
!ls -la /content/cs/runs/calibration_v2 /content/cs/runs/metric_selection
backup()
drive.flush_and_unmount()
print('flushed to Drive: safe to close')

Done. Results are in `MyDrive/countersign/countersign_runs/` (`calibration_v2/`, `metric_selection/`, `logs/`).